# 2.4 Toplama İşlemleri: Min, Max ve Aradaki Her Şey

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/02-numpy/04-aggregates.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Aggregations: Min, Max, and Everything in Between

Herhangi bir veri kümesini keşfetmenin ilk adımı genelde çeşitli özet istatistikleri hesaplamaktır. En yaygın özet istatistikler ortalama ve standart sapmadır; veri kümesindeki "tipik" değerleri özetlerler. Ancak toplam, çarpım, medyan, minimum ve maksimum, çeyrekler (quantile) vb. diğer toplama işlemleri de yararlıdır.

NumPy, diziler üzerinde çalışmak için hızlı yerleşik toplama fonksiyonları sunar; burada birkaçını ele alıp deneyeceğiz.

## Dizideki değerlerin toplanması

Hızlı bir örnek olarak bir dizideki tüm değerlerin toplamını hesaplamayı düşünelim. Python bunu yerleşik sum fonksiyonuyla yapabilir:


In [ ]:
# python_sum.py
import numpy as np
rng = np.random.default_rng()

L = rng.random(100)
print(sum(L))



Sözdizimi NumPy'nin sum fonksiyonuna oldukça benzer; en basit durumda sonuç aynıdır:


In [ ]:
# numpy_sum.py
print(np.sum(L))



Ancak işlem derlenmiş kodda yürütüldüğü için NumPy sürümü çok daha hızlı hesaplanır:


In [ ]:
# timeit_sum.py
import time

big_array = rng.random(1000000)

t0 = time.perf_counter()
sum(big_array)
print(f"Python sum: {(time.perf_counter()-t0)*1000:.1f} ms")

t0 = time.perf_counter()
np.sum(big_array)
print(f"NumPy sum:  {(time.perf_counter()-t0)*1000:.1f} ms")

# IPython'da kitaptaki karşılaştırma:
# %timeit sum(big_array)      → ~89.9 ms
# %timeit np.sum(big_array)   → ~521 µs



Dikkat: sum ile np.sum özdeş değildir; bu bazen kafa karıştırır! Özellikle isteğe bağlı argümanların anlamları farklıdır (sum(x, 1) toplamı 1'den başlatır; np.sum(x, 1) eksen 1 boyunca toplar) ve np.sum çok boyutlu dizileri bilir — bir sonraki bölümde göreceğiz.

> **Not**
>

## Minimum ve maksimum

Benzer şekilde Python'da yerleşik min ve max fonksiyonları vardır; herhangi bir dizinin minimum ve maksimum değerini bulmak için kullanılır:


In [ ]:
# python_minmax.py
print(min(big_array), max(big_array))



NumPy'nin karşılık gelen fonksiyonları benzer sözdizimine sahiptir ve yine çok daha hızlı çalışır:


In [ ]:
# numpy_minmax.py
print(np.min(big_array), np.max(big_array))

# IPython'da kitaptaki karşılaştırma:
# %timeit min(big_array)      → ~72 ms
# %timeit np.min(big_array)   → ~564 µs



min, max, sum ve birkaç diğer NumPy toplaması için daha kısa sözdizimi, dizi nesnesinin kendi yöntemlerini kullanmaktır:


In [ ]:
# dizi_yontemleri.py
print(big_array.min(), big_array.max(), big_array.sum())



Mümkün olduğunca NumPy dizileri üzerinde bu toplama işlemlerinin NumPy sürümünü kullandığınızdan emin olun!

### Çok boyutlu toplama işlemleri

Yaygın toplama türlerinden biri satır veya sütun boyunca toplamadır. Verileriniz iki boyutlu bir dizide saklansın:


In [ ]:
# matris_M.py
M = rng.integers(0, 10, (3, 4))
print(M)



NumPy toplama işlemleri çok boyutlu dizinin tüm elemanlarına uygulanır:


In [ ]:
# matris_toplam.py
print(M.sum())



Toplama fonksiyonları, toplamanın yapılacağı ekseni belirten ek bir argüman alır. Örneğin her sütundaki minimum değeri axis=0 ile bulabiliriz:


In [ ]:
# axis_0.py
print(M.min(axis=0))



Fonksiyon dört sütuna karşılık gelen dört değer döndürür.

Benzer şekilde her satırdaki maksimum değeri bulabiliriz:


In [ ]:
# axis_1.py
print(M.max(axis=1))



Buradaki axis belirtimi başka dillerden gelen kullanıcılar için kafa karıştırıcı olabilir. axis anahtar sözcüğü, döndürülecek boyutu değil, daraltılacak (collapse) boyutu belirtir. Yani axis=0 demek eksen 0 daraltılacak demektir: iki boyutlu dizilerde her sütundaki değerler toplanır.

> **Not**
>

### Diğer toplama fonksiyonları

NumPy benzer API'ye sahip birkaç toplama fonksiyonu daha sunar; çoğunun eksik değerleri (NaN) yok sayan NaN-güvenli karşılığı vardır — eksik değerler IEEE kayan nokta NaN ile işaretlenir (bkz. Eksik Veri bölümü).

NumPy'de yararlı toplama fonksiyonlarının listesi:

Kitabın geri kalanında bu toplama işlemlerini sık göreceksiniz.

## Örnek: ABD başkanlarının ortalama boyu kaç?

NumPy'deki toplama işlemleri bir değer kümesi için özet istatistik görevi görebilir. Küçük bir örnek olarak tüm ABD başkanlarının boylarını ele alalım. Bu veri president_heights.csv dosyasında bulunur; virgülle ayrılmış etiket ve değer listesidir:


```
order,name,height(cm)
1,George Washington,189
2,John Adams,170
3,Thomas Jefferson,189
...
```


Kitapta dosyayı okumak için Pandas kullanılır (Bölüm 3'te ayrıntılı ele alınır). Boylar santimetre cinsindendir. Bu sayfada veriyi doğrudan NumPy dizisi olarak kullanıyoruz (orijinal CSV ile aynı değerler):


In [ ]:
# baskan_boylari.py
import numpy as np

# president_heights.csv — height(cm) sütunu (44 başkan)
heights = np.array([
    189, 170, 189, 163, 183, 171, 185, 168, 173, 183, 173, 173, 175, 178, 183, 193, 178, 173,
    174, 183, 183, 168, 170, 178, 182, 180, 183, 178, 182, 188, 175, 179, 183, 193, 182, 183,
    177, 185, 188, 188, 182, 185, 191, 182
])
print(heights)



> **Not**
>

Veri dizimiz hazır; çeşitli özet istatistikleri hesaplayabiliriz:


In [ ]:
# baskan_ozet.py
print("Ortalama boy:     ", heights.mean())
print("Standart sapma:   ", heights.std())
print("Minimum boy:      ", heights.min())
print("Maksimum boy:     ", heights.max())



Her durumda toplama işlemi tüm diziyi tek bir özet değere indirgedi; dağılım hakkında bilgi verdi. Çeyrekleri (quantile) de hesaplayabiliriz:


In [ ]:
# baskan_ceyrek.py
print("25. yüzdelik:     ", np.percentile(heights, 25))
print("Medyan:           ", np.median(heights))
print("75. yüzdelik:     ", np.percentile(heights, 75))



ABD başkanlarının medyan boyu 182 cm'dir — yaklaşık altı fitten biraz kısa.

Bazen bu verinin görsel temsilini görmek daha yararlıdır; Matplotlib araçlarıyla yapılabilir (Matplotlib Bölüm 4'te ayrıntılı ele alınır). Kitaptaki kod şu grafiği üretir:


In [ ]:
# Matplotlib gerektirir (Jupyter / tam Python ortamı)
# %matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')  # kitapta 'seaborn-whitegrid'

plt.hist(heights)
plt.title('Height Distribution of US Presidents')
plt.xlabel('height (cm)')
plt.ylabel('number')
plt.show()



### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Başkan boyları üzerinde np.mean, np.std ve np.percentile(heights, [25, 50, 75]) hesaplayın; sonuçları yorumlayın:
      
        import numpy as np
heights = np.array([
    189, 170, 189, 163, 183, 171, 185, 168, 173, 183, 173, 173, 175, 178, 183, 193, 178, 173,
    174, 183, 183, 168, 170, 178, 182, 180, 183, 178, 182, 188, 175, 179, 183, 193, 182, 183,
    177, 185, 188, 188, 182, 185, 191, 182
])
print("Ortalama:", heights.mean())
print("Std:", heights.std())
print("Çeyrekler:", np.percentile(heights, [25, 50, 75]))

> **Not**
>
